## Exploring data at a bronze level


- Import nessisery pyspark.

In [0]:
import pyspark as ps
import pyspark.sql.functions as f


- load data 
- checking if it's the right Schema

In [0]:
DATA_PATH = '/Volumes/marathos/default/marathos_raw'
df_marathon = spark.read.csv(DATA_PATH + '/TWO_CENTURIES_OF_UM_RACES.csv', header=True)

df_marathon.limit(100).display()
df_marathon.printSchema()


- Number of columns.
- Number of rows.

In [0]:
n_rows = df_marathon.count()
n_cols = len(df_marathon.columns)

print(f"Number of rows:{n_rows:,}")
print(f"Number of columns:{n_cols}")

In [0]:
numeric_cols = ["Year of event", "Event number of finishers",
                "Athlete year of birth", "Athlete average speed"]

exprs = [f"TRY_CAST(`{c}` AS double) AS `{c}`" for c in numeric_cols]

df_marathon.selectExpr(*exprs).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).display()

In [0]:
from pyspark.sql.functions import desc

df_marathon.groupBy("Athlete country").count()\
    .orderBy(desc("count")).display()

In [0]:
from pyspark.sql.functions import regexp_extract, col, count, when 

units = df_marathon.withColumn(
    "unit",
    regexp_extract(col("event distance/length"), r"([a-zA-z]+)$", 1)
)
units.groupBy("unit").count().orderBy("count", ascending=False).display()

In [0]:
df_marathon.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_marathon.columns
    ]).display()

In [0]:
df_marathon.select("Event name").distinct().count()

In [0]:
ages = df_marathon.withColumn(
    "athlete_age",
    col("Year of event").cast("double") - col("Athlete year of birth")
    .cast("double")
)
ages.select("athlete_age").summary().display()

ages.groupBy("Athlete age category").count() \
    .orderBy("Athlete age category").display()